# Feature Engineering for Churn Prediction

**Goal:** turn raw sales + customer data into governed ML features (RFM + behavior), train a churn model on them, and serve live predictions — every artifact a Unity Catalog object.

The first cells install the feature-engineering library this notebook uses.

In [0]:
# COMMAND ----------

%pip install databricks-feature-engineering

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.1 MB/s  0:00:00
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.122.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-29abfe56-5b5c-4ec6-9ad6-0003f8df38e5
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [databricks-feature-engineering]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# COMMAND ----------

%pip install scikit-learn mlflow

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 104.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 95.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 93.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 98.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.6/826.6 kB 39.3 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-29abfe56-5b5c-4ec6-9ad6-0003f8df38e5
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs

In [0]:
dbutils.library.restartPython()

### Step 1 — Import the toolkit

Spark for transformations, the Feature Engineering client for Unity Catalog feature tables, MLflow + scikit-learn for the model.

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering import FeatureLookup

import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully")

{"ts": "2026-11-26 06:22:09.463", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjcwNzE4Nzk3NDQ1Mjc5OBABIAEyJDAxYTIxNjg0LTBmMmItNzU1OC1iNDdhLWQ5NzBjYzU1MmZhZDokZmUyM2Q5YTQtOWJiZi0zOGYxLWE5MDUtNzYyOWRlOWE1OTNjSgsI45+f2AYQwIXrNlABWAFgAWiIrJXQiMWjDYgBAA==.", "context": {}}
{"ts": "2026-11-26 06:22:09.463", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjcwNzE4Nzk3NDQ1Mjc5OBABIAEyJDAxYTIxNjg0LTBmMmItNzU1OC1iNDdhLWQ5NzBjYzU1MmZhZDokZmUyM2Q5YTQtOWJiZi0zOGYxLWE5MDUtNzYyOWRlOWE1OTNjSgsI45+f2AYQwIXrNlABWAFgAWiIrJXQiMWjDYgBAA==.", "context": {}}
{"ts": "2026-11-26 06:22:09.463", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjcwNzE4Nzk3NDQ1Mjc5OBABIAEyJDAxYTIxNjg0LTBmMmItNzU1OC1iNDdhLWQ5NzBjYzU1MmZhZDokZmUyM2Q5YTQtOWJiZi0zOGYxLWE5MDUtNzYyOWRl

All libraries imported successfully


### Step 2 — Name every Unity Catalog object

One place to see every table, feature table and model this notebook touches — sales, customers, inventory, features, model.

In [0]:
# COMMAND ----------

CATALOG = "retail_ai_prod"

GOLD_SCHEMA = "gold_data"
FEATURE_SCHEMA = "features"
MODEL_SCHEMA = "models"

SALES_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.sales_transactions"
)

CUSTOMER_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.customer_profiles"
)

INVENTORY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.inventory_levels"
)

FEATURE_TABLE = (
    f"{CATALOG}.{FEATURE_SCHEMA}.customer_rfm_features"
)

MODEL_NAME = (
    f"{CATALOG}.{MODEL_SCHEMA}.customer_churn_model"
)

# Fixed snapshot date for reproducible dummy-project results
AS_OF_DATE = "2026-11-15"

print("Sales:", SALES_TABLE)
print("Customers:", CUSTOMER_TABLE)
print("Features:", FEATURE_TABLE)
print("Model:", MODEL_NAME)

Sales: retail_ai_prod.gold_data.sales_transactions
Customers: retail_ai_prod.gold_data.customer_profiles
Features: retail_ai_prod.features.customer_rfm_features
Model: retail_ai_prod.models.customer_churn_model


### Step 3 — Initialize the Feature Engineering client

This client creates, writes and reads Unity Catalog feature tables — features become governed, shareable data assets.

In [0]:
# COMMAND ----------

fe = FeatureEngineeringClient()

print("Feature Engineering Client initialized")

Feature Engineering Client initialized


### Step 4 — Load the gold data

1,000 sales transactions, 200 customer profiles, 96 inventory snapshots — the governed source data everything is built on.

In [0]:
# COMMAND ----------

sales_df = spark.table(SALES_TABLE)

customer_df = spark.table(CUSTOMER_TABLE)

inventory_df = spark.table(INVENTORY_TABLE)

print("Sales:", sales_df.count())
print("Customers:", customer_df.count())
print("Inventory:", inventory_df.count())

display(sales_df.limit(5))
display(customer_df.limit(5))

Sales: 1000
Customers: 200
Inventory: 96


transaction_id,transaction_date,customer_id,product_id,product_name,category,store_city,quantity,gross_sales,discount_amount,returns_amount,net_revenue,payment_method,channel
T000001,2026-03-27,C0026,P1003,Smart Watch,Electronics,Kolkata,3,17997.0,0.0,17997.0,0.0,UPI,Online
T000002,2025-09-29,C0087,P3002,Running Shoes,Fashion,Ahmedabad,2,5998.0,0.0,0.0,5998.0,UPI,Store
T000003,2026-05-15,C0139,P6001,Yoga Mat,Sports,Jaipur,4,3996.0,399.6,1998.0,1598.4,UPI,Online
T000004,2025-04-03,C0112,P6001,Yoga Mat,Sports,Delhi,3,2997.0,149.85,1498.5,1348.65,UPI,Store
T000005,2025-11-29,C0015,P1003,Smart Watch,Electronics,Hyderabad,3,17997.0,0.0,0.0,17997.0,Cash,Store


customer_id,customer_name,city,signup_date,loyalty_tier,preferred_channel,age_band
C0001,Customer 001,Delhi,2022-02-21,Gold,App,26-35
C0002,Customer 002,Mumbai,2022-07-29,Bronze,Store,46-60
C0003,Customer 003,Jaipur,2022-03-03,Bronze,App,26-35
C0004,Customer 004,Jaipur,2025-02-23,Silver,Store,60+
C0005,Customer 005,Ahmedabad,2023-03-28,Platinum,Store,36-45


### Step 5 — Preview the data

A quick look at real transactions before building features from them.

In [0]:
# COMMAND ----------

display(
    sales_df.select(
        "transaction_id",
        "transaction_date",
        "customer_id",
        "product_id",
        "quantity",
        "net_revenue"
    ).orderBy("transaction_date", ascending=False)
)

transaction_id,transaction_date,customer_id,product_id,quantity,net_revenue
T000912,2026-11-12,C0043,P6001,1,949.05
T000896,2026-11-12,C0122,P1003,2,10798.2
T000729,2026-11-12,C0149,P2001,1,3299.0
T000874,2026-11-11,C0158,P2001,1,-494.85
T000773,2026-11-09,C0119,P5001,1,699.0
T000348,2026-11-09,C0187,P6001,3,2997.0
T000086,2026-11-08,C0076,P3001,1,799.0
T000721,2026-11-08,C0191,P6001,2,1698.3
T000512,2026-11-08,C0070,P6001,1,999.0
T000149,2026-11-08,C0108,P2002,2,12998.0


## Calculate customer-level RFM features

In [0]:
# COMMAND ----------

AS_OF_DATE = F.to_date(F.lit(AS_OF_DATE))

sales_aggregated = (
    sales_df
    .groupBy("customer_id")
    .agg(
        F.max("transaction_date").alias("last_purchase_date"),

        F.countDistinct("transaction_id").alias("frequency"),

        F.sum("net_revenue").alias("monetary"),

        F.sum("quantity").alias("total_units_purchased"),

        F.sum("gross_sales").alias("total_gross_sales"),

        F.sum("returns_amount").alias("total_returns"),

        F.countDistinct("category").alias("category_diversity"),

        F.avg("net_revenue").alias("avg_order_value")
    )
)

display(sales_aggregated)

customer_id,last_purchase_date,frequency,monetary,total_units_purchased,total_gross_sales,total_returns,category_diversity,avg_order_value
C0026,2026-03-27,6,27489.449999999997,14,53086.0,24592.0,3,4581.575
C0087,2026-09-29,2,6592.15,3,6697.0,0.0,2,3296.075
C0139,2026-09-20,4,34023.5,14,57386.0,16394.0,3,8505.875
C0112,2026-10-11,8,54414.450000000004,22,69878.0,9445.0,4,6801.806250000001
C0015,2026-10-12,4,33355.25,10,34090.0,0.0,3,8338.8125
C0145,2026-10-14,6,28494.850000000006,16,38084.0,4896.0,4,4749.141666666667
C0037,2026-08-21,8,57576.950000000004,22,72078.0,7748.0,4,7197.118750000001
C0074,2026-07-01,5,4437.75,16,35084.0,26488.0,4,887.55
C0117,2026-01-11,5,11662.35,13,15987.0,3595.5,4,2332.4700000000003
C0092,2026-10-14,9,34347.85,15,45285.0,8198.0,5,3816.4277777777775


### Recency — days since each customer's last purchase

The "R" in RFM: how long ago did we last see each customer?

In [0]:
# COMMAND ----------

sales_aggregated = (
    sales_aggregated
    .withColumn(
        "recency_days",
        F.datediff(
            AS_OF_DATE,
            F.col("last_purchase_date")
        )
    )
)

display(
    sales_aggregated.select(
        "customer_id",
        "last_purchase_date",
        "recency_days"
    )
)

customer_id,last_purchase_date,recency_days
C0026,2026-03-27,233
C0087,2026-09-29,47
C0139,2026-09-20,56
C0112,2026-10-11,35
C0015,2026-10-12,34
C0145,2026-10-14,32
C0037,2026-08-21,86
C0074,2026-07-01,137
C0117,2026-01-11,308
C0092,2026-10-14,32


### Join customers with their purchase history

Left join keeps every customer — even those who never bought anything.

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_df.alias("c")
    .join(
        sales_aggregated.alias("s"),
        on="customer_id",
        how="left"
    )
)

display(customer_features_df)

customer_id,customer_name,city,signup_date,loyalty_tier,preferred_channel,age_band,last_purchase_date,frequency,monetary,total_units_purchased,total_gross_sales,total_returns,category_diversity,avg_order_value,recency_days
C0001,Customer 001,Delhi,2022-02-21,Gold,App,26-35,2026-09-03,7,41755.55,20,83980.0,36691.0,3,5965.078571428572,73
C0002,Customer 002,Mumbai,2022-07-29,Bronze,Store,46-60,2026-03-05,4,20252.999999999996,9,25191.0,2199.0,3,5063.249999999999,255
C0003,Customer 003,Jaipur,2022-03-03,Bronze,App,26-35,2025-09-06,3,5024.15,4,21496.0,15497.0,3,1674.7166666666665,435
C0004,Customer 004,Jaipur,2025-02-23,Silver,Store,60+,2026-02-09,7,46097.3,19,49881.0,0.0,4,6585.328571428572,279
C0005,Customer 005,Ahmedabad,2023-03-28,Platinum,Store,36-45,2026-11-01,5,20743.25,9,24991.0,849.0,3,4148.65,14
C0006,Customer 006,Jaipur,2022-11-23,Platinum,Web,36-45,2026-05-25,5,13500.2,12,16088.0,999.0,3,2700.04,174
C0007,Customer 007,Mumbai,2023-03-17,Gold,App,18-25,2026-11-08,5,26288.75,14,34586.0,6598.0,5,5257.75,7
C0008,Customer 008,Ahmedabad,2022-07-18,Gold,Web,60+,2026-10-21,5,27712.3,10,32490.0,1348.5,4,5542.46,25
C0009,Customer 009,Hyderabad,2022-03-30,Platinum,Store,18-25,2026-06-01,3,1699.3,6,27994.0,24995.0,3,566.4333333333333,167
C0010,Customer 010,Ahmedabad,2022-06-11,Gold,Store,60+,2026-09-16,1,11698.2,2,12998.0,0.0,1,11698.2,60


### Fill the gaps for non-buyers

No purchases? Zeros instead of nulls — and recency set to 9,999 days so the model sees "never bought".

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .fillna({
        "frequency": 0,
        "monetary": 0.0,
        "total_units_purchased": 0,
        "total_gross_sales": 0.0,
        "total_returns": 0.0,
        "category_diversity": 0,
        "avg_order_value": 0.0
    })

    .withColumn(
        "recency_days",
        F.coalesce(
            F.col("recency_days"),
            F.lit(9999)
        )
    )
)

### Behavioral metrics

Return rate, customer tenure and average order value — churn signals beyond plain RFM.

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .withColumn(
        "return_rate",
        F.when(
            F.col("total_gross_sales") > 0,
            F.col("total_returns") /
            F.col("total_gross_sales")
        ).otherwise(0.0)
    )

    .withColumn(
        "customer_tenure_days",
        F.datediff(
            AS_OF_DATE,
            F.col("signup_date")
        )
    )

    .withColumn(
        "avg_order_value",
        F.when(
            F.col("frequency") > 0,
            F.col("monetary") / F.col("frequency")
        ).otherwise(0.0)
    )
)

display(customer_features_df)

customer_id,customer_name,city,signup_date,loyalty_tier,preferred_channel,age_band,last_purchase_date,frequency,monetary,total_units_purchased,total_gross_sales,total_returns,category_diversity,avg_order_value,recency_days,return_rate,customer_tenure_days
C0001,Customer 001,Delhi,2022-02-21,Gold,App,26-35,2026-09-03,7,41755.55,20,83980.0,36691.0,3,5965.078571428572,73,0.4369016432483925,1728
C0002,Customer 002,Mumbai,2022-07-29,Bronze,Store,46-60,2026-03-05,4,20252.999999999996,9,25191.0,2199.0,3,5063.249999999999,255,0.0872930808622127,1570
C0003,Customer 003,Jaipur,2022-03-03,Bronze,App,26-35,2025-09-06,3,5024.15,4,21496.0,15497.0,3,1674.7166666666665,435,0.7209248232229252,1718
C0004,Customer 004,Jaipur,2025-02-23,Silver,Store,60+,2026-02-09,7,46097.3,19,49881.0,0.0,4,6585.328571428572,279,0.0,630
C0005,Customer 005,Ahmedabad,2023-03-28,Platinum,Store,36-45,2026-11-01,5,20743.25,9,24991.0,849.0,3,4148.65,14,0.03397223000280101,1328
C0006,Customer 006,Jaipur,2022-11-23,Platinum,Web,36-45,2026-05-25,5,13500.2,12,16088.0,999.0,3,2700.04,174,0.06209597215315763,1453
C0007,Customer 007,Mumbai,2023-03-17,Gold,App,18-25,2026-11-08,5,26288.75,14,34586.0,6598.0,5,5257.75,7,0.19077083212860696,1339
C0008,Customer 008,Ahmedabad,2022-07-18,Gold,Web,60+,2026-10-21,5,27712.3,10,32490.0,1348.5,4,5542.46,25,0.04150507848568791,1581
C0009,Customer 009,Hyderabad,2022-03-30,Platinum,Store,18-25,2026-06-01,3,1699.3,6,27994.0,24995.0,3,566.4333333333333,167,0.8928699006930056,1691
C0010,Customer 010,Ahmedabad,2022-06-11,Gold,Store,60+,2026-09-16,1,11698.2,2,12998.0,0.0,1,11698.2,60,0.0,1618


### Rank customers into R / F / M quintiles

`ntile(5)` splits customers into five bands by recency, frequency and monetary value — 5 is best.

In [0]:
# COMMAND ----------

recency_window = Window.orderBy(
    F.col("recency_days").asc()
)

frequency_window = Window.orderBy(
    F.col("frequency").desc()
)

monetary_window = Window.orderBy(
    F.col("monetary").desc()
)

customer_features_df = (
    customer_features_df

    .withColumn(
        "r_score",
        F.ntile(5).over(recency_window)
    )

    .withColumn(
        "f_score",
        F.ntile(5).over(frequency_window)
    )

    .withColumn(
        "m_score",
        F.ntile(5).over(monetary_window)
    )

    .withColumn(
        "rfm_score",
        F.col("r_score") +
        F.col("f_score") +
        F.col("m_score")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Segment customers from their RFM scores

Champions, Loyal Customers, At Risk… — the classic marketing segments, computed from the scores.

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df
    .withColumn(
        "customer_segment",

        F.when(
            (F.col("r_score") >= 4) &
            (F.col("f_score") >= 4) &
            (F.col("m_score") >= 4),
            "Champions"
        )

        .when(
            (F.col("r_score") >= 3) &
            (F.col("f_score") >= 3),
            "Loyal Customers"
        )

        .when(
            F.col("r_score") <= 2,
            "At Risk"
        )

        .otherwise("Regular Customers")
    )
)

display(
    customer_features_df.select(
        "customer_id",
        "recency_days",
        "frequency",
        "monetary",
        "rfm_score",
        "customer_segment"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,recency_days,frequency,monetary,rfm_score,customer_segment
C0023,94,7,92484.25,5,Regular Customers
C0022,79,10,79523.2,5,Regular Customers
C0122,3,10,70748.04999999999,3,At Risk
C0155,15,9,63153.15,3,At Risk
C0137,294,8,61663.3,7,Regular Customers
C0037,86,8,57576.950000000004,5,Regular Customers
C0127,258,7,57350.100000000006,8,Regular Customers
C0156,50,9,56727.65000000001,4,At Risk
C0024,20,9,54710.350000000006,3,At Risk
C0176,53,9,54515.75,4,At Risk


### Encode loyalty and channel as numbers

Bronze→1 … Platinum→4, plus a channel score — categorical data turned model-ready.

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .withColumn(
        "loyalty_score",
        F.when(F.col("loyalty_tier") == "Bronze", 1)
        .when(F.col("loyalty_tier") == "Silver", 2)
        .when(F.col("loyalty_tier") == "Gold", 3)
        .when(F.col("loyalty_tier") == "Platinum", 4)
        .otherwise(0)
    )

    .withColumn(
        "channel_score",
        F.when(F.col("preferred_channel") == "App", 1)
        .when(F.col("preferred_channel") == "Web", 2)
        .when(F.col("preferred_channel") == "Store", 3)
        .otherwise(0)
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Assemble the final customer feature table

One row per customer with all model features — recency, RFM values, behavior, loyalty, segment.

In [0]:
# COMMAND ----------

customer_rfm_features = customer_features_df.select(
    "customer_id",

    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",

    "total_units_purchased",
    "total_returns",
    "return_rate",

    "category_diversity",
    "customer_tenure_days",

    "r_score",
    "f_score",
    "m_score",
    "rfm_score",

    "loyalty_score",
    "channel_score",

    "customer_segment"
)

display(customer_rfm_features)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,recency_days,frequency,monetary,avg_order_value,total_units_purchased,total_returns,return_rate,category_diversity,customer_tenure_days,r_score,f_score,m_score,rfm_score,loyalty_score,channel_score,customer_segment
C0023,94,7,92484.25,13212.035714285714,19,9748.5,0.08705494682133576,2,736,3,1,1,5,4,1,Regular Customers
C0022,79,10,79523.2,7952.32,26,1998.0,0.022945999954062064,5,584,3,1,1,5,4,2,Regular Customers
C0122,3,10,70748.05,7074.805,25,14393.0,0.16341754186772636,4,848,1,1,1,3,3,2,At Risk
C0155,15,9,63153.149999999994,7017.016666666666,25,14643.0,0.17840999086201645,6,1569,1,1,1,3,1,3,At Risk
C0137,294,8,61663.3,7707.9125,28,13996.0,0.16608126067970383,5,827,5,1,1,7,4,3,Regular Customers
C0037,86,8,57576.950000000004,7197.118750000001,22,7748.0,0.10749465856433309,4,720,3,1,1,5,4,1,Regular Customers
C0127,258,7,57350.100000000006,8192.871428571429,21,0.0,0.0,4,408,5,2,1,8,1,3,Regular Customers
C0156,50,9,56727.65,6303.072222222222,21,1698.5,0.026099048848322808,5,799,2,1,1,4,4,2,At Risk
C0024,20,9,54710.350000000006,6078.927777777779,25,44741.5,0.4151380190211088,3,1466,1,1,1,3,2,3,At Risk
C0176,53,9,54515.750000000015,6057.3055555555575,22,599.0,0.01000367413741274,5,403,2,1,1,4,3,3,At Risk


### Check the feature table

Row counts, unique IDs, nulls — trust the data before training on it.

In [0]:
# COMMAND ----------

print("Total customers:", customer_rfm_features.count())

print(
    "Unique customer IDs:",
    customer_rfm_features
    .select("customer_id")
    .distinct()
    .count()
)

print(
    "Null customer IDs:",
    customer_rfm_features
    .filter(F.col("customer_id").isNull())
    .count()
)

assert customer_rfm_features.count() == customer_df.count()

assert (
    customer_rfm_features
    .select("customer_id")
    .distinct()
    .count()
    == customer_rfm_features.count()
)

assert (
    customer_rfm_features
    .filter(F.col("customer_id").isNull())
    .count()
    == 0
)

print("Customer feature validation passed")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total customers: 200
Unique customer IDs: 200
Null customer IDs: 0
Customer feature validation passed


### Product sales in the last 90 days

Recent demand per product and city — the base for inventory features.

In [0]:
# COMMAND ----------

sales_90d = (
    sales_df
    .filter(
        F.col("transaction_date") >=
        F.date_sub(AS_OF_DATE, 90)
    )
    .groupBy("product_id", "store_city")
    .agg(
        F.sum("quantity").alias("units_sold_90d"),
        F.sum("net_revenue").alias("revenue_90d")
    )
)

display(sales_90d)

product_id,store_city,units_sold_90d,revenue_90d
P1002,Bengaluru,9,21693.800000000003
P3001,Delhi,10,4194.75
P2001,Hyderabad,4,2804.15
P4002,Mumbai,4,1677.2
P5001,Jaipur,5,3145.5
P1002,Pune,2,6298.2
P4002,Bengaluru,5,2695.5
P4001,Mumbai,6,1258.6
P5001,Ahmedabad,4,2551.35
P3002,Kolkata,3,8547.15


### Build product inventory features

Join recent demand with stock levels — stock cover days, sales velocity and a reorder flag per product and city.

In [0]:
# COMMAND ----------

product_inventory_features = (
    inventory_df.alias("i")
    .join(
        sales_90d.alias("s"),
        on=["product_id", "store_city"],
        how="left"
    )

    .fillna({
        "units_sold_90d": 0,
        "revenue_90d": 0.0
    })

    .withColumn(
        "available_qty",
        F.col("on_hand_qty") - F.col("reserved_qty")
    )

    .withColumn(
        "avg_daily_units_sold",
        F.col("units_sold_90d") / 90.0
    )

    .withColumn(
        "stock_cover_days",
        F.when(
            F.col("avg_daily_units_sold") > 0,
            F.col("available_qty") /
            F.col("avg_daily_units_sold")
        ).otherwise(None)
    )

    .withColumn(
        "reorder_required",
        F.when(
            F.col("available_qty") < F.col("reorder_level"),
            1
        ).otherwise(0)
    )

    .select(
        "product_id",
        "store_city",
        "on_hand_qty",
        "reserved_qty",
        "available_qty",
        "reorder_level",
        "unit_cost",
        "units_sold_90d",
        "revenue_90d",
        "avg_daily_units_sold",
        "stock_cover_days",
        "reorder_required"
    )
)

display(product_inventory_features)

product_id,store_city,on_hand_qty,reserved_qty,available_qty,reorder_level,unit_cost,units_sold_90d,revenue_90d,avg_daily_units_sold,stock_cover_days,reorder_required
P1001,Jaipur,155,2,153,29,1545.96,11,20991.6,0.12222222222222222,1251.8181818181818,0
P1001,Delhi,44,5,39,17,1648.97,6,3248.7000000000003,0.06666666666666667,585.0,0
P1001,Mumbai,13,7,6,28,1361.39,4,-999.6,0.044444444444444446,135.0,1
P1001,Bengaluru,28,10,18,25,1280.59,1,2499.0,0.011111111111111112,1620.0,1
P1001,Hyderabad,74,7,67,24,1307.94,1,0.0,0.011111111111111112,6030.0,0
P1001,Pune,0,0,0,38,1260.41,4,6747.3,0.044444444444444446,0.0,1
P1001,Ahmedabad,164,0,164,31,1708.95,7,16368.45,0.07777777777777778,2108.5714285714284,0
P1001,Kolkata,14,0,14,26,1509.1,4,-1124.5500000000002,0.044444444444444446,315.0,1
P1002,Jaipur,76,1,75,26,1579.06,3,5948.3,0.03333333333333333,2250.0,0
P1002,Delhi,102,0,102,26,1663.2,6,13646.1,0.06666666666666667,1530.0,0


### Validate the product features

Assert one row per product-city — a silent duplicate would poison downstream joins.

In [0]:
# COMMAND ----------

assert (
    product_inventory_features
    .groupBy("product_id", "store_city")
    .count()
    .filter(F.col("count") > 1)
    .count() == 0
)

assert (
    product_inventory_features
    .filter(F.col("available_qty") < 0)
    .count() == 0
)

print("Product inventory features validated")

Product inventory features validated


##Create a synthetic churn label

In [0]:
# COMMAND ----------

CHURN_THRESHOLD_DAYS = 120

churn_labels_df = (
    customer_rfm_features
    .select(
        "customer_id",
        F.when(
            F.col("recency_days") >= CHURN_THRESHOLD_DAYS,
            1
        ).otherwise(0).alias("is_churn_proxy")
    )
)

display(churn_labels_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,is_churn_proxy
C0001,0
C0002,1
C0003,1
C0004,1
C0005,0
C0006,1
C0007,0
C0008,0
C0009,1
C0010,0


### How many customers actually churned?

The proxy label — inactive for 120+ days — and its class balance, the first thing to check before training.

In [0]:
# COMMAND ----------

display(
    churn_labels_df
    .groupBy("is_churn_proxy")
    .count()
    .orderBy("is_churn_proxy")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


is_churn_proxy,count
0,136
1,64


##Register customer features in Unity Catalog

In [0]:
# COMMAND ----------

if not spark.catalog.tableExists(FEATURE_TABLE):

    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["customer_id"],
        df=customer_rfm_features,
        description=(
            "Customer RFM and behavioral features "
            "for churn prediction and segmentation"
        ),
        tags={
            "project": "retail_supply_chain_copilot",
            "feature_group": "customer_rfm"
        }
    )

    print("Customer feature table created")

else:

    fe.write_table(
        name=FEATURE_TABLE,
        df=customer_rfm_features,
        mode="merge"
    )

    print("Customer feature table updated")

2026/11/26 06:22:51 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['customer_id'] of table 'retail_ai_prod.features.customer_rfm_features' to NOT NULL.
2026/11/26 06:22:53 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['customer_id'] on table 'retail_ai_prod.features.customer_rfm_features'.
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
2026/11/26 06:23:00 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'retail_ai_prod.features.customer_rfm_features'.


Customer feature table created


### Register product features in Unity Catalog

Inventory features become a governed, versioned feature table — just like the customer features.

In [0]:
# COMMAND ----------

PRODUCT_FEATURE_TABLE = (
    f"{CATALOG}.{FEATURE_SCHEMA}.product_inventory_features"
)

if not spark.catalog.tableExists(PRODUCT_FEATURE_TABLE):

    fe.create_table(
        name=PRODUCT_FEATURE_TABLE,
        primary_keys=["product_id", "store_city"],
        df=product_inventory_features,
        description=(
            "Product and store inventory features "
            "for replenishment and demand forecasting"
        ),
        tags={
            "project": "retail_supply_chain_copilot",
            "feature_group": "inventory"
        }
    )

else:

    fe.write_table(
        name=PRODUCT_FEATURE_TABLE,
        df=product_inventory_features,
        mode="merge"
    )

print("Product inventory feature table ready")

2026/11/26 06:23:06 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['product_id', 'store_city'] of table 'retail_ai_prod.features.product_inventory_features' to NOT NULL.
2026/11/26 06:23:09 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['product_id', 'store_city'] on table 'retail_ai_prod.features.product_inventory_features'.
2026/11/26 06:23:16 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'retail_ai_prod.features.product_inventory_features'.


Product inventory feature table ready


### Read features back through the client

`fe.read_table()` resolves the registered tables — the same features any team or job can now reuse.

In [0]:
# COMMAND ----------

registered_customer_features = fe.read_table(
    name=FEATURE_TABLE
)

registered_inventory_features = fe.read_table(
    name=PRODUCT_FEATURE_TABLE
)

display(registered_customer_features)
display(registered_inventory_features)

customer_id,recency_days,frequency,monetary,avg_order_value,total_units_purchased,total_returns,return_rate,category_diversity,customer_tenure_days,r_score,f_score,m_score,rfm_score,loyalty_score,channel_score,customer_segment
C0023,94,7,92484.25,13212.035714285714,19,9748.5,0.08705494682133576,2,736,3,1,1,5,4,1,Regular Customers
C0022,79,10,79523.2,7952.32,26,1998.0,0.022945999954062064,5,584,3,1,1,5,4,2,Regular Customers
C0122,3,10,70748.05,7074.805,25,14393.0,0.16341754186772636,4,848,1,1,1,3,3,2,At Risk
C0155,15,9,63153.149999999994,7017.016666666666,25,14643.0,0.17840999086201645,6,1569,1,1,1,3,1,3,At Risk
C0137,294,8,61663.3,7707.9125,28,13996.0,0.16608126067970383,5,827,5,1,1,7,4,3,Regular Customers
C0037,86,8,57576.950000000004,7197.118750000001,22,7748.0,0.10749465856433309,4,720,3,1,1,5,4,1,Regular Customers
C0127,258,7,57350.100000000006,8192.871428571429,21,0.0,0.0,4,408,5,2,1,8,1,3,Regular Customers
C0156,50,9,56727.65,6303.072222222222,21,1698.5,0.026099048848322808,5,799,2,1,1,4,4,2,At Risk
C0024,20,9,54710.350000000006,6078.927777777779,25,44741.5,0.4151380190211088,3,1466,1,1,1,3,2,3,At Risk
C0176,53,9,54515.750000000015,6057.3055555555575,22,599.0,0.01000367413741274,5,403,2,1,1,4,3,3,At Risk


product_id,store_city,on_hand_qty,reserved_qty,available_qty,reorder_level,unit_cost,units_sold_90d,revenue_90d,avg_daily_units_sold,stock_cover_days,reorder_required
P1001,Jaipur,155,2,153,29,1545.96,11,20991.6,0.12222222222222222,1251.8181818181818,0
P1001,Delhi,44,5,39,17,1648.97,6,3248.7000000000003,0.06666666666666667,585.0,0
P1001,Mumbai,13,7,6,28,1361.39,4,-999.6,0.044444444444444446,135.0,1
P1001,Bengaluru,28,10,18,25,1280.59,1,2499.0,0.011111111111111112,1620.0,1
P1001,Hyderabad,74,7,67,24,1307.94,1,0.0,0.011111111111111112,6030.0,0
P1001,Pune,0,0,0,38,1260.41,4,6747.3,0.044444444444444446,0.0,1
P1001,Ahmedabad,164,0,164,31,1708.95,7,16368.45,0.07777777777777778,2108.5714285714284,0
P1001,Kolkata,14,0,14,26,1509.1,4,-1124.5500000000002,0.044444444444444446,315.0,1
P1002,Jaipur,76,1,75,26,1579.06,3,5948.3,0.03333333333333333,2250.0,0
P1002,Delhi,102,0,102,26,1663.2,6,13646.1,0.06666666666666667,1530.0,0


### Query features with plain SQL

Once in Unity Catalog, features are just tables — any SQL tool can join them. Here: segment sizes and their RFM averages.

In [0]:
%sql
-- COMMAND ----------

SELECT
    customer_segment,
    COUNT(*) AS customers,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    ROUND(AVG(frequency), 2) AS avg_frequency
FROM retail_ai_prod.features.customer_rfm_features
GROUP BY customer_segment
ORDER BY customers DESC;

customer_segment,customers,avg_monetary,avg_frequency
At Risk,80,25153.27,5.39
Loyal Customers,48,21091.69,4.23
Regular Customers,40,34234.26,7.13
Champions,32,5829.85,2.53


## Create a training dataset using FeatureLookup

In [0]:
# COMMAND ----------

from databricks.feature_engineering import FeatureLookup

print("FeatureLookup imported")

FeatureLookup imported


### Define the FeatureLookups

Point at the feature table by name — the platform joins each feature to the right customer at training time.

In [0]:
# COMMAND ----------

feature_lookups = [

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="frequency"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="monetary"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="avg_order_value"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="total_units_purchased"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="total_returns"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="return_rate"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="category_diversity"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="customer_tenure_days"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="loyalty_score"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="channel_score"
    )
]

print("Feature lookups defined:", len(feature_lookups))

2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING databricks.ml_features.entities.feature_lookup: The feature_name parameter is deprecated. Use "feature_names".
2026/11/26 06:23:28 WARNING

Feature lookups defined: 10


### Define the training spine

Just customer_id + churn label — the features arrive automatically from the feature table via the lookups.

In [0]:
# COMMAND ----------

training_spine_df = churn_labels_df.select(
    "customer_id",
    "is_churn_proxy"
)

training_set = fe.create_training_set(
    df=training_spine_df,

    feature_lookups=feature_lookups,

    label="is_churn_proxy",

    exclude_columns=["customer_id"]
)

print("Training set created successfully")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Training set created successfully


### Materialize the training dataset

`load_df()` joins the spine with the feature table — point-in-time correct, no manual joins.

In [0]:
# COMMAND ----------

training_spark_df = training_set.load_df()

display(training_spark_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


frequency,monetary,avg_order_value,total_units_purchased,total_returns,return_rate,category_diversity,customer_tenure_days,loyalty_score,channel_score,is_churn_proxy
7,41755.55,5965.078571428572,20,36691.0,0.4369016432483925,3,1728,3,1,0
4,20252.999999999996,5063.249999999999,9,2199.0,0.0872930808622127,3,1570,1,3,1
3,5024.15,1674.7166666666665,4,15497.0,0.7209248232229252,3,1718,1,1,1
7,46097.3,6585.328571428572,19,0.0,0.0,4,630,2,3,1
5,20743.25,4148.65,9,849.0,0.03397223000280101,3,1328,4,3,0
5,13500.2,2700.04,12,999.0,0.06209597215315763,3,1453,4,2,1
5,26288.75,5257.75,14,6598.0,0.19077083212860696,5,1339,3,1,0
5,27712.3,5542.46,10,1348.5,0.04150507848568791,4,1581,3,2,0
3,1699.3,566.4333333333333,6,24995.0,0.8928699006930056,3,1691,4,3,1
1,11698.2,11698.2,2,0.0,0.0,1,1618,3,3,0


### Training set health check

Rows, missing labels, missing features — confirm the automatic joins produced complete training data.

In [0]:
# COMMAND ----------

print("Training rows:", training_spark_df.count())

print(
    "Missing labels:",
    training_spark_df
    .filter(F.col("is_churn_proxy").isNull())
    .count()
)

print(
    "Missing feature rows:",
    training_spark_df
    .filter(F.col("frequency").isNull())
    .count()
)

display(
    training_spark_df
    .groupBy("is_churn_proxy")
    .count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Training rows: 200
Missing labels: 0
Missing feature rows: 0


is_churn_proxy,count
1,64
0,136


## Train the churn prediction model

In [0]:
# COMMAND ----------

training_pdf = training_spark_df.toPandas()

print("Pandas shape:", training_pdf.shape)

display(training_pdf.head())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Pandas shape: (200, 11)


frequency,monetary,avg_order_value,total_units_purchased,total_returns,return_rate,category_diversity,customer_tenure_days,loyalty_score,channel_score,is_churn_proxy
7,41755.55,5965.078571428572,20,36691.0,0.4369016432483925,3,1728,3,1,0
4,20252.999999999996,5063.249999999999,9,2199.0,0.0872930808622127,3,1570,1,3,1
3,5024.15,1674.7166666666665,4,15497.0,0.7209248232229252,3,1718,1,1,1
7,46097.3,6585.328571428572,19,0.0,0.0,4,630,2,3,1
5,20743.25,4148.65,9,849.0,0.03397223000280101,3,1328,4,3,0


### Select features and label

10 numeric features, one binary churn label — the exact contract the model will remember.

In [0]:
# COMMAND ----------

FEATURE_COLUMNS = [
    "frequency",
    "monetary",
    "avg_order_value",
    "total_units_purchased",
    "total_returns",
    "return_rate",
    "category_diversity",
    "customer_tenure_days",
    "loyalty_score",
    "channel_score"
]

TARGET_COLUMN = "is_churn_proxy"

X = training_pdf[FEATURE_COLUMNS].astype(float)

y = training_pdf[TARGET_COLUMN].astype(int)

print("X shape:", X.shape)
print("Y shape:", y.shape)

print("Churn distribution:")
print(y.value_counts())

X shape: (200, 10)
Y shape: (200,)
Churn distribution:
is_churn_proxy
0    136
1     64
Name: count, dtype: int64


### Split train / test

Hold out 25% of customers (stratified) to measure real generalization.

In [0]:
# COMMAND ----------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.25,

    random_state=42,

    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 150
Testing rows: 50


### Train the churn model

RandomForest with balanced class weights — churners are the minority, so they count more during training.

In [0]:
# COMMAND ----------

churn_model = RandomForestClassifier(

    n_estimators=100,

    max_depth=5,

    min_samples_split=4,

    min_samples_leaf=2,

    class_weight="balanced",

    random_state=42
)

churn_model.fit(
    X_train,
    y_train
)

print("Churn model training completed!")

Churn model training completed!


### Predict on the test set

Hard churn labels plus a churn probability for every held-out customer.

In [0]:
# COMMAND ----------

y_pred = churn_model.predict(X_test)

y_probability = churn_model.predict_proba(X_test)[:, 1]

print("Predictions completed")

Predictions completed


### Evaluate the model

Accuracy, precision, recall, F1 and ROC AUC — one line each, on the held-out customers.

In [0]:
# COMMAND ----------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)

Accuracy: 0.72
Precision: 0.5833333333333334
Recall: 0.4375
F1 Score: 0.5
ROC AUC: 0.6709558823529411


### Which features drive churn?

Feature importances, ranked — the "explain it to the business" view of the model.

In [0]:
# COMMAND ----------

importance_df = pd.DataFrame({

    "feature": FEATURE_COLUMNS,

    "importance": churn_model.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

feature,importance
avg_order_value,0.18535036920450637
monetary,0.1699394624248695
total_units_purchased,0.12655014792939975
return_rate,0.12618920109833204
customer_tenure_days,0.11932405970832846
total_returns,0.08211693621130188
frequency,0.08201113871524603
channel_score,0.048925297403137516
loyalty_score,0.03337319147330334
category_diversity,0.02622019583157512


##Log the model using MLflow and Feature Engineering

In [0]:
# COMMAND ----------

assert training_pdf[FEATURE_COLUMNS].notnull().all().all(), (
    "Missing features detected. Fix the feature pipeline "
    "before model training."
)

X = training_pdf[FEATURE_COLUMNS].astype(float)
y = training_pdf[TARGET_COLUMN].astype(int)

print("Training features validated without additional transformations")

Training features validated without additional transformations


### Point MLflow at Unity Catalog

Experiment tracking and the model registry both live in UC — runs land in `/Shared/retail_customer_churn_experiment`.

In [0]:
# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")

mlflow.set_experiment(
    "/Shared/retail_customer_churn_experiment"
)

print("MLflow configured")

2026/11/26 06:23:48 INFO mlflow.tracking.fluent: Experiment with name '/Shared/retail_customer_churn_experiment' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


MLflow configured


### Log the run — params, metrics, model

Every training run is tracked: hyperparameters, evaluation metrics and the model artifact, in one reproducible bundle.

In [0]:
# COMMAND ----------

with mlflow.start_run(
    run_name="retail_customer_churn_rf"
) as run:

    # Log hyperparameters

    mlflow.log_params({
        "algorithm": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 5,
        "churn_threshold_days": CHURN_THRESHOLD_DAYS,
        "test_size": 0.25,
        "random_state": 42
    })

    # Log evaluation metrics

    mlflow.log_metrics({
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc)
    })

    # Log model with feature metadata

    fe.log_model(
        model=churn_model,

        artifact_path="model",

        flavor=mlflow.sklearn,

        training_set=training_set,

        registered_model_name=MODEL_NAME,

        await_registration_for=300,

        infer_input_example=True
    )

    print("MLflow run ID:", run.info.run_id)

print("Model logged and registered!")

2026/11/26 06:23:49 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
2026/11/26 06:23:57 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-77fbc76d-f022.cloud.databricks.com/ml/experiments/2707187974452842/models/m-7b87ea344de34363ad52a345bdd92550?o=7474654159918600
2026/11/26 06:23:58 INFO mlflow.pyfunc: Validating input example against model signature
2026/11/26 06:23:58 WARNING mlflow.models.model

Uploading artifacts:   0%|          | 0/16 [00:00<?, ?it/s]

🔗 Created version '1' of model 'retail_ai_prod.models.customer_churn_model': https://dbc-77fbc76d-f022.cloud.databricks.com/explore/data/models/retail_ai_prod/models/customer_churn_model/version/1?o=7474654159918600


MLflow run ID: 5800aded07bc4ea4acebdee5321cec05
Model logged and registered!


### List the registered model versions

Query the UC Model Registry — every version, its status and the run that produced it.

In [0]:
# COMMAND ----------

from mlflow.tracking import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
    f"name='{MODEL_NAME}'"
)

for version in model_versions:

    print("Model:", version.name)
    print("Version:", version.version)
    print("Status:", version.status)
    print("Run ID:", version.run_id)

Model: retail_ai_prod.models.customer_churn_model
Version: 1
Status: READY
Run ID: 5800aded07bc4ea4acebdee5321cec05


##Perform inference using automatic feature lookup

In [0]:
# COMMAND ----------

model_versions = client.search_model_versions(
    f"name='{MODEL_NAME}'"
)

latest_version = max(
    model_versions,
    key=lambda x: int(x.version)
).version

MODEL_URI = f"models:/{MODEL_NAME}/{latest_version}"

print("Model URI:", MODEL_URI)

Model URI: models:/retail_ai_prod.models.customer_churn_model/1


### Choose customers to score

Any list of customer IDs works — here we take the first 10 from the customer table.

In [0]:
# COMMAND ----------

inference_df = (
    spark.table(CUSTOMER_TABLE)
    .select("customer_id")
    .limit(10)
)

display(inference_df)

customer_id
C0001
C0002
C0003
C0004
C0005
C0006
C0007
C0008
C0009
C0010


### Batch scoring with automatic feature lookup

`fe.score_batch()` loads the model, fetches each customer's features from the feature table and scores — zero manual joins.

In [0]:
# COMMAND ----------

predictions_df = fe.score_batch(
    model_uri=MODEL_URI,
    df=inference_df
)

display(predictions_df)

2026/11/26 06:24:10 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/11/26 06:24:10 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


customer_id,frequency,monetary,avg_order_value,total_units_purchased,total_returns,return_rate,category_diversity,customer_tenure_days,loyalty_score,channel_score,prediction
C0001,7,41755.55,5965.078571428572,20,36691.0,0.4369016432483925,3,1728,3,1,0.0
C0002,4,20252.999999999996,5063.249999999999,9,2199.0,0.0872930808622127,3,1570,1,3,0.0
C0003,3,5024.15,1674.7166666666665,4,15497.0,0.7209248232229252,3,1718,1,1,1.0
C0004,7,46097.3,6585.328571428572,19,0.0,0.0,4,630,2,3,0.0
C0005,5,20743.25,4148.65,9,849.0,0.03397223000280101,3,1328,4,3,0.0
C0006,5,13500.2,2700.04,12,999.0,0.06209597215315763,3,1453,4,2,1.0
C0007,5,26288.75,5257.75,14,6598.0,0.19077083212860696,5,1339,3,1,0.0
C0008,5,27712.3,5542.46,10,1348.5,0.04150507848568791,4,1581,3,2,0.0
C0009,3,1699.3,566.4333333333333,6,24995.0,0.8928699006930056,3,1691,4,3,1.0
C0010,1,11698.2,11698.2,2,0.0,0.0,1,1618,3,3,0.0


### Turn predictions into business language

"Potentially Inactive" vs "Active" — the same numbers a store manager would act on.

In [0]:
# COMMAND ----------

business_predictions = (
    predictions_df
    .withColumn(
        "churn_status",
        F.when(
            F.col("prediction") == 1,
            "Potentially Inactive"
        ).otherwise("Active")
    )
    .select(
        "customer_id",
        "prediction",
        "churn_status"
    )
)

display(business_predictions)

customer_id,prediction,churn_status
C0001,0.0,Active
C0002,0.0,Active
C0003,1.0,Potentially Inactive
C0004,0.0,Active
C0005,0.0,Active
C0006,1.0,Potentially Inactive
C0007,0.0,Active
C0008,0.0,Active
C0009,1.0,Potentially Inactive
C0010,0.0,Active


##Create a Feature Serving Endpoint

In [0]:
# COMMAND ----------

from databricks.feature_engineering import (
    FeatureEngineeringClient,
    FeatureLookup
)

FEATURE_SPEC_NAME = (
    "retail_ai_prod.features.customer_rfm_serving_spec"
)

serving_features = [

    FeatureLookup(
        table_name=FEATURE_TABLE,

        lookup_key="customer_id",

        feature_names=[
            "frequency",
            "monetary",
            "avg_order_value",
            "total_units_purchased",
            "total_returns",
            "return_rate",
            "category_diversity",
            "customer_tenure_days",
            "loyalty_score",
            "channel_score"
        ]
    )
]

try:

    fe.create_feature_spec(
        name=FEATURE_SPEC_NAME,
        features=serving_features
    )

    print("FeatureSpec created")

except Exception as e:

    if "already exists" in str(e).lower():

        print("FeatureSpec already exists")

    else:

        raise

FeatureSpec created


### Create the feature serving endpoint

Publish the FeatureSpec as `retail-customer-features` — a small, scale-to-zero online endpoint for low-latency feature lookups.

In [0]:
# COMMAND ----------

from databricks.feature_engineering.entities.feature_serving_endpoint import (
    ServedEntity,
    EndpointCoreConfig
)

FEATURE_ENDPOINT_NAME = "retail-customer-features"

endpoint_config = EndpointCoreConfig(

    served_entities=ServedEntity(

        feature_spec_name=FEATURE_SPEC_NAME,

        workload_size="Small",

        scale_to_zero_enabled=True
    )
)

feature_endpoint = fe.create_feature_serving_endpoint(

    name=FEATURE_ENDPOINT_NAME,

    config=endpoint_config
)

print("Feature Serving endpoint creation submitted")
print(feature_endpoint)

Feature Serving endpoint creation submitted
<FeatureServingEndpoint: creation_time_millis=1795674288000, creator='sdemanpreetsingh@gmail.com', name='retail-customer-features', state={'config_update': 'IN_PROGRESS',
 'ready': 'NOT_READY',
 'suspend': 'NOT_SUSPENDED',
 'system_update_failure': False}, url='https://dbc-77fbc76d-f022.cloud.databricks.com/serving-endpoints/retail-customer-features/invocations'>


### Diagnostics — client capabilities and endpoint state

Inspect what the Feature Engineering client exposes (online stores, specs, endpoints) and the live endpoint state. Kept from debugging — safe to skip in the demo.

In [0]:
import inspect
import databricks.feature_engineering as dfe
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()
print("FE package version:", getattr(dfe, "__version__", "Not exposed"))

for name in [
    "list_online_stores",
    "get_online_store",
    "create_online_store",
    "publish_table",
    "get_feature_spec",
    "get_feature_serving_endpoint",
    "create_feature_serving_endpoint",
    "update_feature_serving_endpoint",
]:
    present = hasattr(fe, name)
    sig = None
    if present:
        try:
            sig = str(inspect.signature(getattr(fe, name)))
        except Exception as e:
            sig = f"<signature error: {e}>"
    print(f"{name}: present={present} | sig={sig}")

print("\n--- Offline Table Detail ---")
display(spark.sql("DESCRIBE DETAIL retail_ai_prod.features.customer_rfm_features"))

print("\n--- Catalog Table Checks ---")
for tbl in [
    "retail_ai_prod.features.customer_rfm_features",
    "retail_ai_prod.features.customer_rfm_features_online",
    "retail_ai_prod.features.product_inventory_features",
]:
    print(f"{tbl} exists:", spark.catalog.tableExists(tbl))

print("\n--- Online Stores ---")
try:
    print("list_online_stores():", fe.list_online_stores())
except Exception as e:
    print("list_online_stores() error:", repr(e))

print("\n--- Feature Spec Check ---")
try:
    if hasattr(fe, "get_feature_spec"):
        print(fe.get_feature_spec(name="retail_ai_prod.features.customer_rfm_serving_spec"))
    else:
        print("get_feature_spec not on client; checking via SQL DESCRIBE EXTENDED...")
        display(spark.sql("DESCRIBE EXTENDED retail_ai_prod.features.customer_rfm_serving_spec"))
except Exception as e:
    print("Feature spec check error:", repr(e))

print("\n--- Endpoint State & Config ---")
try:
    ep = fe.get_feature_serving_endpoint(name="retail-customer-features")
    print("state:", ep.state)
    print("ready:", ep.state.get("ready") if isinstance(ep.state, dict) else None)
    print("config_update:", ep.state.get("config_update") if isinstance(ep.state, dict) else None)
    print("config:", getattr(ep, "config", ep))
except Exception as e:
    print("Endpoint check error:", repr(e))

FE package version: Not exposed
list_online_stores: present=True | sig=() -> List[databricks.ml_features.entities.online_store.DatabricksOnlineStore]
get_online_store: present=True | sig=(*, name: str) -> Optional[databricks.ml_features.entities.online_store.DatabricksOnlineStore]
create_online_store: present=True | sig=(*, name: str, capacity: str, read_replica_count: Optional[int] = None, usage_policy_id: Optional[str] = None) -> databricks.ml_features.entities.online_store.DatabricksOnlineStore
publish_table: present=True | sig=(*, online_store: Union[databricks.ml_features.online_store_spec.online_store_spec.OnlineStoreSpec, databricks.ml_features.entities.online_store.DatabricksOnlineStore], source_table_name: str = None, online_table_name: str = None, publish_mode: str = 'TRIGGERED', filter_condition: Optional[str] = None, mode: str = 'merge', streaming: bool = False, checkpoint_location: Optional[str] = None, trigger: Dict[str, Any] = {'processingTime': '5 minutes'}, features: U

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,45c7149d-d67d-47ae-902a-7cf198162146,retail_ai_prod.features.customer_rfm_features,Customer RFM and behavioral features for churn prediction and segmentation,,2026-11-26T06:22:49.921Z,2026-11-26T06:22:59.000Z,List(),List(),1,11080,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4ccb7a1e-4a7f-4236-b95a-56f856e428de, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-574eb161-a798-4109-be3d-480790fd230b, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true



--- Catalog Table Checks ---
retail_ai_prod.features.customer_rfm_features exists: True
retail_ai_prod.features.customer_rfm_features_online exists: False
retail_ai_prod.features.product_inventory_features exists: True

--- Online Stores ---
list_online_stores(): []

--- Feature Spec Check ---
get_feature_spec not on client; checking via SQL DESCRIBE EXTENDED...


{"ts": "2026-11-26 06:24:51.780", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[TABLE_OR_VIEW_NOT_FOUND] The table or view `retail_ai_prod`.`features`.`customer_rfm_serving_spec` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 18;\n'DescribeRelation true, [col_name#20493, data_type#20494, comment#20495]\n+- 'UnresolvedTableOrView [retail_ai_prod, features, customer_rfm_serving_spec], DESCRIBE TABLE, tr

Feature spec check error: AnalysisException("[TABLE_OR_VIEW_NOT_FOUND] The table or view `retail_ai_prod`.`features`.`customer_rfm_serving_spec` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 18;\n'DescribeRelation true, [col_name#20493, data_type#20494, comment#20495]\n+- 'UnresolvedTableOrView [retail_ai_prod, features, customer_rfm_serving_spec], DESCRIBE TABLE, true, QueryLike, true\n")

--- Endpoint State & Config ---
state: {'ready': 'NOT_READY', 'config_update': 'IN_PROGRESS', 'suspend': 'NOT_SUSPENDED', 'system_update_failure': False}
ready: NOT_READY
config_update: IN_PROGRESS
config: <FeatureServingEndpoint

### Inspect feature table metadata and online stores

Table metadata, online store status and the serving endpoint's readiness — the operational view of the feature pipeline.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

print("=== 1. Offline Table Metadata (SQL) ===")
desc_df = spark.sql("DESCRIBE EXTENDED retail_ai_prod.features.customer_rfm_features")
desc_df.show(truncate=False)

detail_df = spark.sql("DESCRIBE DETAIL retail_ai_prod.features.customer_rfm_features")
detail_df.select("id", "name", "format", "location", "properties").show(truncate=False)

print("\n=== 2. Online Stores List ===")
try:
    stores = fe.list_online_stores()
    print("Online stores count:", len(stores))
    for s in stores:
        print("Store:", s)
except Exception as e:
    print("list_online_stores() failed:", repr(e))

print("\n=== 3. Feature Serving Endpoint State ===")
try:
    ep = fe.get_feature_serving_endpoint(name="retail-customer-features")
    print("Endpoint name:", getattr(ep, "name", None))
    print("Endpoint state:", getattr(ep, "state", None))
    if isinstance(getattr(ep, "state", None), dict):
        print("ready:", ep.state.get("ready"))
        print("config_update:", ep.state.get("config_update"))
except Exception as e:
    print("get_feature_serving_endpoint() failed:", repr(e))

=== 1. Offline Table Metadata (SQL) ===
+------------------------+---------+-------+
|col_name                |data_type|comment|
+------------------------+---------+-------+
|customer_id             |string   |NULL   |
|recency_days            |int      |NULL   |
|frequency               |bigint   |NULL   |
|monetary                |double   |NULL   |
|avg_order_value         |double   |NULL   |
|total_units_purchased   |bigint   |NULL   |
|total_returns           |double   |NULL   |
|return_rate             |double   |NULL   |
|category_diversity      |bigint   |NULL   |
|customer_tenure_days    |int      |NULL   |
|r_score                 |int      |NULL   |
|f_score                 |int      |NULL   |
|m_score                 |int      |NULL   |
|rfm_score               |int      |NULL   |
|loyalty_score           |int      |NULL   |
|channel_score           |int      |NULL   |
|customer_segment        |string   |NULL   |
|# Clustering Information|         |       |
|# col_name    

### Score real customers through a live endpoint

3 customers straight from the feature table → churn probability from the `retail-churn-classifier` Model Serving endpoint — the exact API a store app would call.

In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")
ENDPOINT_NAME = "retail-churn-classifier"   # live Model Serving endpoint (the retail-customer-features Feature Serving endpoint is currently aborted)

FEATURES = [
    "frequency", "monetary", "avg_order_value", "total_units_purchased",
    "total_returns", "return_rate", "category_diversity",
    "customer_tenure_days", "loyalty_score", "channel_score",
]

# take 3 real customers and their feature values from the Unity Catalog feature table
sample_customers = (
    spark.table("retail_ai_prod.features.customer_rfm_features")
    .select("customer_id", *FEATURES)
    .limit(3)
    .toPandas()
)
print("Scoring customers:", list(sample_customers["customer_id"]))

try:
    response = deploy_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={
            "dataframe_split": sample_customers[FEATURES].to_dict(orient="split")
        },
    )
    for cid, risk in zip(sample_customers["customer_id"], response.predictions):
        print(f"{cid}: churn risk {risk:.0%} → {'HIGH — contact this customer' if risk > 0.5 else 'low'}")
except Exception as e:
    print("Predict failed with error:")
    print(repr(e))

Scoring customers: ['C0023', 'C0022', 'C0122']
Predict failed with error:
HTTPError('404 Client Error: The given endpoint does not exist, please retry after checking the specified model and version deployment exists. for url: https://dbc-77fbc76d-f022.cloud.databricks.com/serving-endpoints/retail-churn-classifier/invocations. Response text: {"error_code": "ENDPOINT_NOT_FOUND", "message": "The given endpoint does not exist, please retry after checking the specified model and version deployment exists."}')


## Create a product inventory feature lookup

In [0]:
# COMMAND ----------
# One-time setup for Feature Serving: a Databricks-managed online store
# (Lakebase-backed) plus a SNAPSHOT publish of the feature table.
# The serving endpoint looks features up in the ONLINE store — this is
# the step that was missing when the endpoint update failed with
# "No suitable online store found".

import time

ONLINE_STORE_NAME = "retail-online-store"   # Lakebase project id: lowercase, digits, hyphens only
ONLINE_CATALOG    = "retail_online_store"
ONLINE_TABLE_NAME = f"{ONLINE_CATALOG}.default.customer_rfm_features_online"

# 1. Create the online store (reused if it already exists)
online_store = fe.get_online_store(name=ONLINE_STORE_NAME)
if online_store is None:
    online_store = fe.create_online_store(
        name=ONLINE_STORE_NAME,
        capacity="CU_1"           # smallest capacity — enough for the demo
    )
    print("Online store creation submitted")
else:
    print("Online store already exists")

# 2. Wait for the store to become AVAILABLE (first provisioning takes a few minutes)
MAX_WAIT_SECONDS = 600
POLL_INTERVAL   = 20
start = time.time()
while True:
    state = getattr(online_store.state, "name", str(online_store.state)).upper()
    print(f"[{int(time.time()-start)}s] online store state: {state}")
    if state == "AVAILABLE":
        break
    if time.time() - start > MAX_WAIT_SECONDS:
        raise TimeoutError(
            f"Online store not AVAILABLE after {MAX_WAIT_SECONDS}s (state: {state})"
        )
    time.sleep(POLL_INTERVAL)
    online_store = fe.get_online_store(name=ONLINE_STORE_NAME)

# 3. The online table lives in its own UC catalog backed by the online
#    store's Lakebase database — create the catalog if it does not exist yet
spark.sql(f"CREATE CATALOG IF NOT EXISTS {ONLINE_CATALOG}")

# 4. Publish the feature table (one-time SNAPSHOT sync; TRIGGERED or
#    CONTINUOUS mode would need Change Data Feed enabled on the source table)
fe.publish_table(
    online_store=online_store,
    source_table_name=FEATURE_TABLE,
    online_table_name=ONLINE_TABLE_NAME,
    publish_mode="SNAPSHOT",
    # NOTE: the `features` parameter is not supported for Databricks online
    # stores — the whole feature table is published (primary key included).
)
print(f"Published '{FEATURE_TABLE}' -> '{ONLINE_TABLE_NAME}'")
print("Next: run the endpoint-update retry cell, then re-run the poll cell.")

Online store already exists
[0s] online store state: AVAILABLE
Published 'retail_ai_prod.features.customer_rfm_features' -> 'retail_online_store.default.customer_rfm_features_online'
Next: run the endpoint-update retry cell, then re-run the poll cell.


### Wait until the endpoint is READY

First deployment builds a compute container; this polls every 20 s until READY (10-minute cap).

In [0]:
# COMMAND ----------
# Poll until the Feature Serving endpoint is READY
import time

MAX_WAIT_SECONDS = 600   # 10 minutes
POLL_INTERVAL   = 20

start = time.time()
while True:
    endpoint = fe.get_feature_serving_endpoint(name=FEATURE_ENDPOINT_NAME)
    ready    = endpoint.state.get("ready", "")
    cfg_upd  = endpoint.state.get("config_update", "")

    print(f"[{int(time.time()-start)}s] ready={ready}  config_update={cfg_upd}")

    if ready == "READY":
        print("\nEndpoint is READY.")
        break

    if cfg_upd == "UPDATE_FAILED":
        raise RuntimeError(
            f"Endpoint update FAILED. Full state: {endpoint.state}"
        )

    if time.time() - start > MAX_WAIT_SECONDS:
        raise TimeoutError(
            f"Endpoint not ready after {MAX_WAIT_SECONDS}s. State: {endpoint.state}"
        )

    time.sleep(POLL_INTERVAL)

[0s] ready=NOT_READY  config_update=IN_PROGRESS
[20s] ready=NOT_READY  config_update=IN_PROGRESS
[40s] ready=NOT_READY  config_update=IN_PROGRESS
[60s] ready=NOT_READY  config_update=IN_PROGRESS
[80s] ready=NOT_READY  config_update=IN_PROGRESS
[100s] ready=NOT_READY  config_update=IN_PROGRESS
[120s] ready=NOT_READY  config_update=IN_PROGRESS
[140s] ready=NOT_READY  config_update=IN_PROGRESS
[160s] ready=NOT_READY  config_update=IN_PROGRESS
[180s] ready=NOT_READY  config_update=IN_PROGRESS
[201s] ready=NOT_READY  config_update=IN_PROGRESS
[221s] ready=NOT_READY  config_update=IN_PROGRESS
[241s] ready=NOT_READY  config_update=IN_PROGRESS
[261s] ready=NOT_READY  config_update=IN_PROGRESS
[281s] ready=NOT_READY  config_update=IN_PROGRESS
[301s] ready=READY  config_update=NOT_UPDATING

Endpoint is READY.


In [0]:
# COMMAND ----------
# Retry the Feature Serving endpoint deployment after the poll cell raised
# UPDATE_FAILED. This SDK version has no fe.update_feature_serving_endpoint,
# so the update is re-submitted through the Databricks SDK instead.
# NOTE: the re-deployment still requires the feature table
# (retail_ai_prod.features.customer_rfm_features) to be published to an
# online store first — without that, this update will fail the same way.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput

workspace = WorkspaceClient()

workspace.serving_endpoints.update_config(
    name=FEATURE_ENDPOINT_NAME,
    served_entities=[
        ServedEntityInput(
            name="customer_rfm_serving_spec",
            entity_name=FEATURE_SPEC_NAME,
            scale_to_zero_enabled=True,
            workload_size="Small"
        )
    ]
)

print(f"Endpoint update re-submitted for '{FEATURE_ENDPOINT_NAME}'.")
print("Re-run the poll cell above to wait for READY.")

Endpoint update re-submitted for 'retail-customer-features'.
Re-run the poll cell above to wait for READY.
